# 02: LangChain, the Building Blocks

**12 components, one section each**

This notebook follows the [LangChain API map](assets/langchain_api_landscape.png) component by component. Each section covers one component and runs its most useful methods as examples. Anything else in that component is listed at the end of its section so you know it exists.

| # | Component | Examples we run |
|---|---|---|
| 1 | Chat models | `init_chat_model()`, `.invoke()`, `.bind_tools()`, `.with_structured_output()` |
| 2 | Messages | `HumanMessage`, `AIMessage`, `ToolMessage` |
| 3 | Prompts | `ChatPromptTemplate` |
| 4 | Runnables · LCEL | `a \| b \| c`, `RunnableParallel`, `RunnablePassthrough` |
| 5 | Output parsers | `StrOutputParser` |
| 6 | Tools | `@tool` |
| 7 | Retrieval · RAG | `Document`, `.as_retriever()` |
| 8 | Agents | `create_agent()` |
| 9 | Middleware · hooks | `@before_model`, `@wrap_tool_call` |
| 10 | Built-in middleware | `HumanInTheLoopMiddleware`, `ToolCallLimitMiddleware` |
| 11 | Callbacks · tracing | `tags`, `metadata` |
| 12 | Ops utilities | `set_llm_cache()`, `InMemoryCache` |

## Setup (run once)

The LangSmith key is **optional** here, so press Enter to skip it. If you set it, every run in this notebook is traced for free.

Everything runs on `gpt-4o-mini`, so it is cheap (a full run of this notebook costs a few cents). Swap `MODEL` for any provider you like.

In [1]:
%pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters langgraph pydantic

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
import os, getpass

def set_env(var: str):
    """Ask for a secret only if it is not already in the environment."""
    if not os.environ.get(var):
        val = getpass.getpass(f"{var} (Enter to skip): ")
        if val:
            os.environ[var] = val

set_env("OPENAI_API_KEY")
set_env("LANGSMITH_API_KEY")

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "meetup-langchain"
    print("LangSmith tracing ON -> project: meetup-langchain")
else:
    print("LangSmith tracing OFF")

MODEL = "openai:gpt-4o-mini"   # provider:model - change freely

LangSmith tracing ON -> project: meetup-langchain


## 1. Chat models
`init_chat_model()` returns a **chat model**: messages in, an `AIMessage` out. The same methods below work for every provider.

- `init_chat_model()` creates the model from a `provider:model` string
- `.invoke()` sends input and returns the full answer
- `.bind_tools()` tells the model which functions it may ask you to call
- `.with_structured_output()` makes the model return a validated Python object

`temperature=0` gives near-deterministic output, which is good for demos.

**Also worth knowing:** `.stream()`, `.batch()`, `.ainvoke()`, `.astream_events()`, `.with_fallbacks()`, `.with_retry()`, `BaseChatModel`, `usage_metadata`

In [4]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL, temperature=0)

resp = llm.invoke("Explain LangChain in one sentence.")
print("TEXT :", resp.content)
print("TYPE :", type(resp).__name__)

TEXT : LangChain is a framework designed to facilitate the development of applications that leverage large language models by providing tools for chaining together various components like prompts, memory, and APIs.
TYPE : AIMessage


In [5]:
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It is 31°C and sunny in {city}."   # fake data

llm_with_tools = llm.bind_tools([get_weather])   # a typed function with a docstring is enough
ai = llm_with_tools.invoke("What's the weather in Pune?")
print("Model asked for:", ai.tool_calls)         # the model only asks; it never runs the function

Model asked for: [{'name': 'get_weather', 'args': {'city': 'Pune'}, 'id': 'call_GfGO48QvTfSyC7D9zfFpeToU', 'type': 'tool_call'}]


In [4]:
from typing import Literal
from pydantic import BaseModel, Field

class Ticket(BaseModel):
    """A support ticket extracted from a user message."""
    category: Literal["bug", "feature", "question"]
    priority: int = Field(description="1 (low) to 5 (urgent)")
    summary: str = Field(description="At most 12 words")

structured_llm = llm.with_structured_output(Ticket)      # .with_structured_output(): schema in, validated object out
ticket = structured_llm.invoke("The checkout page 500s for every user since the deploy. Revenue is down!")
print(ticket)
print(ticket.model_dump())

category='bug' priority=5 summary='Checkout page returns 500 error for all users'
{'category': 'bug', 'priority': 5, 'summary': 'Checkout page returns 500 error for all users'}


## 2. Messages
Chat models speak in **messages**. These three cover a whole conversation:

- `HumanMessage`: what the user said
- `AIMessage`: what the model said (it can also carry tool calls)
- `ToolMessage`: the result of a tool, sent back to the model

LLMs are stateless, so "memory" is just the list of messages you send again each time.

**Also worth knowing:** `SystemMessage`, `AIMessageChunk`, `.tool_calls`, `.content_blocks`, `trim_messages()`, `filter_messages()`

In [5]:
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage("Hi, my name is Priya."),
    AIMessage("Nice to meet you, Priya!"),
    HumanMessage("What is my name?"),
]
print(llm.invoke(history).content)

Your name is Priya.


In [9]:
from langchain_core.messages import ToolMessage

question = HumanMessage("What's the weather in Pune?")
ai = llm_with_tools.invoke([question])                              # an AIMessage that contains a tool call
# print(ai)
call = ai.tool_calls[0]
# print(call)
result = get_weather(**call["args"])                                # you run the function
# print(result)
tool_msg = ToolMessage(content=result, tool_call_id=call["id"])     # and wrap the result for the model

print(llm_with_tools.invoke([question, ai, tool_msg]).content)

The weather in Pune is currently 31°C and sunny.


## 3. Prompts
`ChatPromptTemplate` is a reusable, parameterised prompt. It separates *what you ask* from *the data you inject*, and its output is just a list of messages, so no model is called yet.

**Also worth knowing:** `.from_messages()`, `PromptTemplate`, `MessagesPlaceholder`, `FewShotChatMessagePromptTemplate`, `.partial()`

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise."),
    ("human", "Explain {topic} to {audience}."),
])

filled = prompt.invoke({"role": "teacher", "topic": "recursion", "audience": "a student"})
print(filled.to_messages())   # just messages, no LLM call yet


[SystemMessage(content='You are a teacher. Be concise.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain recursion to a student.', additional_kwargs={}, response_metadata={})]


## 4. Runnables · LCEL
Prompts, models, parsers and retrievers are all **Runnables**. The **LangChain Expression Language (LCEL)** connects them:

- `a | b | c` pipes the output of one step into the next
- `RunnableParallel` runs branches at the same time and returns a dict
- `RunnablePassthrough` forwards the input unchanged (or adds keys with `.assign()`)

**Also worth knowing:** `RunnableSequence`, `.assign()`, `RunnableLambda`, `RunnableBranch`, `@chain`, `RunnableConfig`, `.pick()`, `.with_config()`

In [10]:
chain = prompt | llm         # a | b | c : each step feeds the next

msg = chain.invoke({"role": "chef", "topic": "caramelisation", "audience": "a teenager"})
print(msg.content)
print("\nChain type:", type(chain).__name__)

Caramelization is the process of cooking sugar until it melts and turns brown, creating a rich, sweet flavor. When you heat sugar, it breaks down and transforms, changing color and taste. This is what gives caramel its delicious, complex flavor, and it's used in desserts like caramel sauce, candies, and even some savory dishes. It's all about heat and patience!

Chain type: RunnableSequence


`RunnablePassthrough` on its own simply returns what it receives. Inside a `RunnableParallel` it lets you keep the original input next to new results.

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

echo = RunnableParallel(original=RunnablePassthrough(), length=lambda s: len(s))
print(echo.invoke("langchain"))   # {'original': 'langchain', 'length': 9}

{'original': 'langchain', 'length': 9}


In [12]:
from langchain_core.output_parsers import StrOutputParser   # explained in section 5

pros = ChatPromptTemplate.from_template("List 3 short pros of {topic}.") | llm | StrOutputParser()
cons = ChatPromptTemplate.from_template("List 3 short cons of {topic}.") | llm | StrOutputParser()
verdict_prompt = ChatPromptTemplate.from_template(
    "Pros:\n{pros}\n\nCons:\n{cons}\n\nGive a one-sentence balanced verdict."
)

debate = (
    RunnableParallel(pros=pros, cons=cons)                                           # both run at once
    | RunnablePassthrough.assign(verdict=verdict_prompt | llm | StrOutputParser())   # keep pros/cons, add a verdict
)

out = debate.invoke({"topic": "remote work"})
print("PROS:\n", out["pros"], "\n\nCONS:\n", out["cons"])
print("\nVERDICT:", out["verdict"])

PROS:
 1. **Flexibility**: Remote work allows employees to create their own schedules, leading to better work-life balance and the ability to manage personal commitments more effectively.

2. **Cost Savings**: Both employees and employers can save money; employees save on commuting and daily expenses, while employers can reduce overhead costs associated with maintaining physical office spaces.

3. **Access to a Wider Talent Pool**: Companies can hire talent from anywhere in the world, increasing diversity and allowing them to find the best candidates regardless of geographic location. 

CONS:
 1. **Isolation**: Remote work can lead to feelings of loneliness and disconnection from colleagues, which may affect morale and collaboration.

2. **Distractions**: Working from home can introduce various distractions, such as household chores, family members, or noise, which can impact productivity.

3. **Work-Life Balance**: The blurred boundaries between work and personal life can make it chal

## 5. Output parsers
A chat model returns an `AIMessage`. `StrOutputParser` unwraps it into a plain Python string, so the end of a chain is easy to use.

For typed data, prefer `.with_structured_output()` from section 1 over parsing text yourself.

**Also worth knowing:** `JsonOutputParser`, `PydanticOutputParser`

In [13]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

text = chain.invoke({"role": "poet", "topic": "the ocean", "audience": "a child"})
print(type(text).__name__, "->", text)

TextAccessor -> The ocean is a big, blue blanket that covers much of our Earth. It's filled with water, waves, and many creatures like fish, dolphins, and colorful coral. Sometimes it sparkles in the sun and can be calm or wild. It’s a place for adventure, where you can splash, explore, and discover the wonders of nature!


## 6. Tools
`@tool` turns a Python function into a tool: a name, a description (from the docstring) and an argument schema (from the type hints). The model reads those to decide when to call it.

**Also worth knowing:** `BaseTool`, `StructuredTool`, `args_schema`, `ToolRuntime`, `InjectedToolArg`, `return_direct`, `ToolException`

In [18]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It is 31°C and sunny in {city}."          # fake data

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

print("name:       ", multiply.name)
print("description:", multiply.description)
print("schema:     ", multiply.args)
print("direct call:", multiply.invoke({"a": 12.5, "b": 8}))

name:        multiply
description: Multiply two numbers.
schema:      {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
direct call: 100.0


### The manual tool-call loop
This is what an agent does under the hood: **model asks → you execute → you send the result back → model answers.** It uses `.bind_tools()` (section 1) and `ToolMessage` (section 2) together with `@tool`.

Keep this cell on screen. In section 8 one call replaces it, and in the LangGraph notebook we turn this exact loop into a graph.

In [19]:
tools = [get_weather, multiply]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

messages = [HumanMessage("What is 12.5 times 8, and what's the weather in Hyderabad?")]

ai = llm_with_tools.invoke(messages)                          # 1) model decides
print("TOOL CALLS:", [(c["name"], c["args"]) for c in ai.tool_calls])
messages.append(ai)

for call in ai.tool_calls:                                    # 2) YOU execute
    messages.append(tools_by_name[call["name"]].invoke(call)) # invoking a tool with a tool call returns a ToolMessage

final = llm_with_tools.invoke(messages)                       # 3) model writes the answer
print("\nFINAL:", final.content)

TOOL CALLS: [('multiply', {'a': 12.5, 'b': 8}), ('get_weather', {'city': 'Hyderabad'})]

FINAL: 12.5 times 8 is 100.0. 

The weather in Hyderabad is currently 31°C and sunny.


## 7. Retrieval · RAG
Retrieval lets the model answer from *your* data. The ingestion half is: load → split → embed → store.

- `Document` is the unit of data: text plus metadata
- `.as_retriever()` turns a vector store into a retriever you can drop into a chain

We use in-memory documents so the demo works without any files. Swap in a document loader for real data.

**Also worth knowing:** `RecursiveCharacterTextSplitter`, `init_embeddings()`, `InMemoryVectorStore`, `.similarity_search()`, `BaseRetriever`, document loaders

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore

# In real life:  docs = PyPDFLoader("handbook.pdf").load()   (pip install langchain-community pypdf)
raw_docs = [
    Document(page_content=(
        "The Hyderabad AI Meetup runs on the last Saturday of every month at the T-Hub campus. "
        "Doors open at 10:00 AM and talks start at 10:30 AM. Free coffee is provided by sponsors. "
        "Registration is required and capped at 150 people."), metadata={"source": "faq", "topic": "logistics"}),
    Document(page_content=(
        "Speakers can submit a talk proposal through the community form. Talks are 20 minutes plus 5 minutes of Q&A. "
        "Proposals about LangChain, LangGraph, evaluation and production LLM systems are especially welcome. "
        "Lightning talks of 5 minutes are open to first-time speakers."), metadata={"source": "cfp", "topic": "speakers"}),
    Document(page_content=(
        "LangGraph models an application as a graph: nodes are functions, edges decide what runs next, and a shared state "
        "is passed along. Checkpointers save the state after every step, which enables memory, human approval and replay."),
        metadata={"source": "notes", "topic": "langgraph"}),
    Document(page_content=(
        "LangSmith records a trace of every run: prompts, model outputs, tool calls, latency, tokens and cost. "
        "Datasets and evaluators let you compare prompt versions before shipping."),
        metadata={"source": "notes", "topic": "langsmith"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
print(f"{len(raw_docs)} docs -> {len(chunks)} chunks")

embeddings = init_embeddings("openai:text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
for d in retriever.invoke("How long is a talk?"):
    print("-", d.metadata, "|", d.page_content[:90], "...")

### A RAG chain
Retrieve → stuff into the prompt → answer **only** from context. The dict at the front is auto-wrapped in a `RunnableParallel`, and `RunnablePassthrough` forwards the question untouched (section 4 again).

In [ ]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. If the answer is not there, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What time do talks start and how long is each talk?"))
print("---")
print(rag_chain.invoke("What is the price of a sponsor booth?"))   # should say it doesn't know

## 8. Agents
`create_agent()` is one call that replaces the manual loop from section 6. The model now **decides** whether to retrieve, which tool to use and when to stop. We expose the retriever as a tool: that is "agentic RAG".

Compare with the RAG chain in section 7: the chain *always* retrieves. The agent retrieves *only if it needs to*, and can retrieve multiple times.

**Also worth knowing:** `system_prompt`, `tools=[...]`, `response_format`, `ToolStrategy`, `ProviderStrategy`, `context_schema`, `checkpointer`, `store`, `AgentState`

In [ ]:
from langchain.agents import create_agent

@tool
def search_knowledge_base(query: str) -> str:
    """Search the meetup/LangChain knowledge base for relevant facts."""
    return format_docs(vector_store.similarity_search(query, k=2))

agent = create_agent(
    model=MODEL,
    tools=[search_knowledge_base, get_weather, multiply],
    system_prompt=(
        "You are a helpful meetup assistant. Use the knowledge base for meetup, LangGraph "
        "and LangSmith questions. Use the other tools for weather and math. Be concise."
    ),
)

result = agent.invoke({"messages": [{"role": "user", "content":
    "When does the meetup start, and what is the weather in Hyderabad? Also what is 15 * 15?"}]})

for m in result["messages"]:
    m.pretty_print()

## 9. Middleware · hooks
Middleware lets you run your own code at fixed points of the agent loop without rewriting it. Two hooks to know:

- `@before_model`: runs before every model call (log, trim, inspect state)
- `@wrap_tool_call`: wraps every tool call (log, retry, catch errors)

Below, the second hook turns a crashing tool (`10 / 0`) into a normal `ToolMessage` so the agent can explain the problem instead of failing.

**Also worth knowing:** `AgentMiddleware`, `@before_agent`, `@after_model`, `@after_agent`, `@wrap_model_call`, `@dynamic_prompt`, `ModelRequest`, `ModelResponse`

In [26]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model, wrap_tool_call
from langchain_core.messages import ToolMessage
from langgraph.runtime import Runtime

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

@before_model
def log_model_call(state: AgentState, runtime: Runtime) -> None:
    print(f"[before_model] sending {len(state['messages'])} messages to the model")

@wrap_tool_call
def safe_tool_call(request, handler):
    print(f"[wrap_tool_call] {request.tool_call['name']}({request.tool_call['args']})")
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(content=f"Tool failed: {e}", tool_call_id=request.tool_call["id"])

hooked_agent = create_agent(
    model=MODEL,
    tools=[divide, multiply],
    system_prompt=(
        "You must use the divide tool for every division. "
        "Do not calculate division yourself, even if the math looks invalid."
    ),
    middleware=[log_model_call, safe_tool_call],
)

res = hooked_agent.invoke({"messages": [{"role": "user", "content": "Use the divide tool. What is 10 divided by 0?"}]})
print("\nFINAL:", res["messages"][-1].content)

[before_model] sending 1 messages to the model
[wrap_tool_call] divide({'a': 10, 'b': 0})
[before_model] sending 3 messages to the model

FINAL: The operation of dividing by zero is undefined and cannot be performed. If you have any other questions or need assistance, feel free to ask!


## 10. Built-in middleware
Common agent behaviours ship ready-made. Two to know:

- `HumanInTheLoopMiddleware`: pause before chosen tools and wait for a human to approve, edit or reject
- `ToolCallLimitMiddleware`: stop runaway tool loops after N calls

Pausing needs saved state, so the agent gets a checkpointer (this is LangGraph persistence; we go deep in notebook 04).

**Also worth knowing:** `SummarizationMiddleware`, `ModelCallLimitMiddleware`, `PIIMiddleware`, `ModelFallbackMiddleware`, `ModelRetryMiddleware`, `ToolRetryMiddleware`, `TodoListMiddleware`, `LLMToolSelectorMiddleware`, `ContextEditingMiddleware`

In [27]:
from langchain.agents.middleware import HumanInTheLoopMiddleware, ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def send_email(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}."          # fake side effect

guarded_agent = create_agent(
    model=MODEL,
    tools=[send_email, get_weather],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True, "get_weather": False}),
        ToolCallLimitMiddleware(run_limit=3),
    ],
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "email-1"}}
paused = guarded_agent.invoke(
    {"messages": [{"role": "user", "content": "Email bob@example.com that the meetup starts at 10:30."}]}, cfg
)
print("PAUSED, waiting for a human:", paused["__interrupt__"][0].value)

PAUSED, waiting for a human: {'action_requests': [{'name': 'send_email', 'args': {'to': 'bob@example.com', 'body': 'The meetup starts at 10:30.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'bob@example.com', 'body': 'The meetup starts at 10:30.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}


In [28]:
# ... the human clicks "Approve" (could be a different process, hours later) ...
resumed = guarded_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg)
print(resumed["messages"][-1].content)

I have sent an email to bob@example.com informing him that the meetup starts at 10:30.


## 11. Callbacks · tracing
Every run can carry **tags** (labels such as `prod` or `experiment-3`) and **metadata** (anything else, such as `user_id` or `app_version`). Pass them in `config`. When LangSmith tracing is on they make traces searchable (see notebook 03).

To see them without LangSmith we use a tiny callback handler that prints what it receives. Callbacks hook into the start, end and error events of models, tools and chains.

**Also worth knowing:** `BaseCallbackHandler`, `config={'callbacks': [...]}`, `run_name`, `collect_runs()`, `LangChainTracer`

In [34]:
from langchain_core.callbacks import BaseCallbackHandler

class ShowTagsAndMetadata(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, *, tags=None, metadata=None, **kwargs):
        wanted = {k: v for k, v in (metadata or {}).items() if k in ("user_id", "app_version")}
        print("tags    :", tags)
        print("metadata:", wanted)

llm.invoke(
    "Say hello in three words.",
    config={
        "tags": ["meetup", "demo"],
        "metadata": {"user_id": "u-42", "app_version": "1.0.3"},
        "callbacks": [ShowTagsAndMetadata()],
    },
)

tags    : ['meetup', 'demo']
metadata: {'user_id': 'u-42', 'app_version': '1.0.3'}


AIMessage(content='Hello, how are you?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 13, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_80c9d72b86', 'id': 'chatcmpl-EemPPfWmEsphbpObRrbnW54dyHlfx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18e3f-d933-7142-9c9a-6005ebf767da-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 6, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 12. Ops utilities
These are production extras rather than day-one tools. We run the most useful one: **caching** returns an identical prompt instantly, for free.

**Also worth knowing:** `InMemoryRateLimiter`, `max_concurrency`, `astream_events v2`, `get_buffer_string()`. For retries and fallbacks, see `.with_retry()` and `.with_fallbacks()` in section 1.

In [32]:
import time
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

set_llm_cache(InMemoryCache())

q = "Name three uses of embeddings in one line each."
for attempt in (1, 2):
    t0 = time.perf_counter()
    llm.invoke(q)
    print(f"attempt {attempt}: {time.perf_counter() - t0:.2f}s")   # the second one is served from the cache

set_llm_cache(None)   # turn caching back off

attempt 1: 2.32s
attempt 2: 0.00s


---
## Recap: what to remember

1. **Chat models + messages + prompts** are the basic calls: `init_chat_model()`, `.invoke()`, `HumanMessage`, `ChatPromptTemplate`.
2. **`a | b | c`** connects Runnables; `StrOutputParser` ends the chain; `.with_structured_output()` beats parsing prose.
3. **Tools** = `@tool` + `.bind_tools()` + `ToolMessage`. An **agent** (`create_agent()`) = model + tools + loop.
4. **RAG** = `Document` → split → embed → store → `.as_retriever()` → prompt.
5. **Middleware** (`@before_model`, `@wrap_tool_call`, `HumanInTheLoopMiddleware`, `ToolCallLimitMiddleware`) adds control without rewriting the loop.
6. **`tags` and `metadata`** make every run searchable once tracing is on.

**Next:** `03_langsmith.ipynb` → see inside those runs and *measure* them.

### Optional exercises
- Add a `get_time(timezone)` tool to the agent in section 8.
- Make `Ticket` in section 1 include a `suggested_assignee` field.
- Reject the email in section 10 with `{"type": "reject", "message": "Not yet"}` instead of approving it.